# Hurtownia danych – Online Retail

## 0. Import bibliotek

In [1]:
import pandas as pd
from datetime import datetime, date

## 1. Wczytanie danych

In [7]:
df = pd.read_csv("Online_Retail.csv", encoding="ISO-8859-1")
print(f"Liczba wierszy przed czyszczeniem: {len(df)}")
df.head()

Liczba wierszy przed czyszczeniem: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## 2. Czyszczenie danych

Dane wymagają oczyszczenia przed załadowaniem do hurtowni, ponieważ zawierają brakujące wartości, anulowane transakcje oraz nieprawidłowe wartości ilości i ceny. Zgodnie z zasadą "garbage in = garbage out", błędne dane prowadziłyby do błędnych raportów analitycznych. Każdy krok czyszczenia jest udokumentowany wraz z liczbą usuniętych rekordów.

In [9]:
# Krok 1: usuń wiersze bez CustomerID
df = df.dropna(subset=["CustomerID"])
print(f"Po usunięciu braków CustomerID: {len(df)} wierszy")

# Krok 2: usuń anulowane transakcje (InvoiceNo zaczyna się od 'C')
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"Po usunięciu anulacji: {len(df)} wierszy")

# Krok 3: usuń rekordy z Quantity <= 0
df = df[df["Quantity"] > 0]
print(f"Po usunięciu Quantity <= 0: {len(df)} wierszy")

# Krok 4: usuń rekordy z UnitPrice <= 0
df = df[df["UnitPrice"] > 0]
print(f"Po usunięciu UnitPrice <= 0: {len(df)} wierszy")

# Krok 5: konwersja dat
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%y %H:%M")

# Krok 6: usuń duplikaty
df = df.drop_duplicates()
print(f"Po usunięciu duplikatów: {len(df)} wierszy")

# Krok 7: dodaj kolumnę Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Upewnij się, że CustomerID jest całkowitoliczbowy
df["CustomerID"] = df["CustomerID"].astype(int)

print("\nPodgląd danych po czyszczeniu:")
df.head()

Po usunięciu braków CustomerID: 406829 wierszy
Po usunięciu anulacji: 397924 wierszy
Po usunięciu Quantity <= 0: 397924 wierszy
Po usunięciu UnitPrice <= 0: 397884 wierszy
Po usunięciu duplikatów: 392692 wierszy

Podgląd danych po czyszczeniu:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## 3. Wybór ziarna (grain)

**Decyzja:** Ziarno zdefiniowano na poziomie pojedynczej pozycji faktury (InvoiceNo + StockCode), czyli jeden wiersz w tabeli faktów odpowiada sprzedaży jednego produktu w ramach jednej faktury.

**Uzasadnienie:** Takie ziarno zapewnia największą szczegółowość danych, co umożliwia analizę sprzedaży na poziomie poszczególnych produktów, klientów i dat. Dane można zawsze agregować do wyższych poziomów (cała faktura, dzień, miesiąc), lecz nie można ich dezagregować, jeśli zostaną wstępnie zagregowane. Przykład analizy, którą umożliwia: "które produkty generują największy przychód w poszczególnych krajach i miesiącach?".

## 4. Schemat gwiazdy

Zaprojektowany schemat zawiera:
- **FactSales** – tabela faktów (miary: Quantity, Revenue, UnitPrice)
- **DimCustomer** – wymiar klienta (CustomerID, Country)
- **DimProduct** – wymiar produktu (StockCode, Description)
- **DimDate** – wymiar czasu (Date, Year, Month, Day, DayOfWeek, Quarter)

Każdy wymiar posiada sztuczny klucz główny (surrogate key). Tabela faktów łączy się z wymiarami wyłącznie przez klucze sztuczne.

## 5. Budowa wymiarów

### Klucze naturalne i sztuczne

| Wymiar | Klucz naturalny | Klucz sztuczny (surrogate key) |
|--------|----------------|--------------------------------|
| DimCustomer | CustomerID | CustomerKey |
| DimProduct  | StockCode   | ProductKey  |
| DimDate     | InvoiceDate (data) | DateKey |

Klucze sztuczne są potrzebne, by uniezależnić hurtownię od systemów źródłowych – naturalny klucz może się zmieniać lub być ponownie użyty, co mogłoby naruszyć integralność historycznych danych.

In [10]:
# ── DimCustomer ──────────────────────────────────────────────────────────────
dim_customer = (
    df[["CustomerID", "Country"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
# Klucz sztuczny
dim_customer.insert(0, "CustomerKey", dim_customer.index + 1)

print("DimCustomer (pierwsze wiersze):")
print(dim_customer.head())
print(f"Liczba klientów: {len(dim_customer)}")

DimCustomer (pierwsze wiersze):
   CustomerKey  CustomerID         Country
0            1       17850  United Kingdom
1            2       13047  United Kingdom
2            3       12583          France
3            4       13748  United Kingdom
4            5       15100  United Kingdom
Liczba klientów: 4346


In [11]:
# ── DimProduct ───────────────────────────────────────────────────────────────
dim_product = (
    df[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .reset_index(drop=True)
)
# Klucz sztuczny
dim_product.insert(0, "ProductKey", dim_product.index + 1)

print("DimProduct (pierwsze wiersze):")
print(dim_product.head())
print(f"Liczba produktów: {len(dim_product)}")

DimProduct (pierwsze wiersze):
   ProductKey StockCode                          Description
0           1    85123A   WHITE HANGING HEART T-LIGHT HOLDER
1           2     71053                  WHITE METAL LANTERN
2           3    84406B       CREAM CUPID HEARTS COAT HANGER
3           4    84029G  KNITTED UNION FLAG HOT WATER BOTTLE
4           5    84029E       RED WOOLLY HOTTIE WHITE HEART.
Liczba produktów: 3665


In [12]:
# ── DimDate ──────────────────────────────────────────────────────────────────
unique_dates = df["InvoiceDate"].dt.date.unique()
dim_date = pd.DataFrame({"Date": pd.to_datetime(unique_dates)})
dim_date = dim_date.sort_values("Date").reset_index(drop=True)

# Atrybuty czasowe
dim_date["Year"]      = dim_date["Date"].dt.year
dim_date["Month"]     = dim_date["Date"].dt.month
dim_date["Day"]       = dim_date["Date"].dt.day
dim_date["DayOfWeek"] = dim_date["Date"].dt.day_name()
dim_date["Quarter"]   = dim_date["Date"].dt.quarter

# Klucz sztuczny (format YYYYMMDD)
dim_date.insert(0, "DateKey", dim_date["Date"].dt.strftime("%Y%m%d").astype(int))

print("DimDate (pierwsze wiersze):")
print(dim_date.head())
print(f"Liczba dat: {len(dim_date)}")

DimDate (pierwsze wiersze):
    DateKey       Date  Year  Month  Day  DayOfWeek  Quarter
0  20101201 2010-12-01  2010     12    1  Wednesday        4
1  20101202 2010-12-02  2010     12    2   Thursday        4
2  20101203 2010-12-03  2010     12    3     Friday        4
3  20101205 2010-12-05  2010     12    5     Sunday        4
4  20101206 2010-12-06  2010     12    6     Monday        4
Liczba dat: 305


## 6. Budowa tabeli faktów

Tabela faktów FactSales łączy się z każdym wymiarem poprzez klucze sztuczne (surrogate keys). Miary to Quantity, UnitPrice oraz Revenue (Quantity × UnitPrice). Tabela nie zawiera kluczy naturalnych – są one przechowywane wyłącznie w wymiarach.

In [13]:
# Przygotuj klucz daty w df
df["DateKey"] = df["InvoiceDate"].dt.strftime("%Y%m%d").astype(int)

# Dołącz klucze sztuczne z wymiarów
fact_sales = df.merge(dim_customer[["CustomerKey", "CustomerID", "Country"]],
                      on=["CustomerID", "Country"], how="left")
fact_sales = fact_sales.merge(dim_product[["ProductKey", "StockCode"]],
                              on="StockCode", how="left")

# Wybierz tylko potrzebne kolumny do tabeli faktów
fact_sales = fact_sales[[
    "InvoiceNo",    # klucz naturalny faktury (do identyfikacji transakcji)
    "CustomerKey",
    "ProductKey",
    "DateKey",
    "Quantity",
    "UnitPrice",
    "Revenue"
]].reset_index(drop=True)

# Klucz sztuczny tabeli faktów
fact_sales.insert(0, "SalesKey", fact_sales.index + 1)

print("FactSales (pierwsze wiersze):")
print(fact_sales.head())
print(f"\nLiczba wierszy w tabeli faktów: {len(fact_sales)}")
print(f"Łączny przychód: {fact_sales['Revenue'].sum():,.2f}")

FactSales (pierwsze wiersze):
   SalesKey InvoiceNo  CustomerKey  ProductKey   DateKey  Quantity  UnitPrice  \
0         1    536365            1           1  20101201         6       2.55   
1         2    536365            1           2  20101201         6       3.39   
2         3    536365            1           3  20101201         8       2.75   
3         4    536365            1           4  20101201         6       3.39   
4         5    536365            1           5  20101201         6       3.39   

   Revenue  
0    15.30  
1    20.34  
2    22.00  
3    20.34  
4    20.34  

Liczba wierszy w tabeli faktów: 392692
Łączny przychód: 8,887,208.89


## 7. SCD – DimCustomer (Typ 2)

**Wybór:** Zastosowano SCD Typ 2 (wersjonowanie historii) dla wymiaru DimCustomer.

**Uzasadnienie:** Klient może zmienić kraj (np. przenieść siedzibę lub zmienić kraj dostawy). Dzięki SCD2 zachowujemy historię zmian – historyczne faktury nadal wskazują na wersję rekordu klienta aktualną w momencie transakcji, co zapewnia poprawność raportów historycznych. Alternatywny SCD1 (nadpisywanie) skutkowałby utratą informacji o poprzednim stanie.

**Dodane kolumny:** `valid_from`, `valid_to`, `is_current`.

In [14]:
# Budowa DimCustomer_SCD2
# Bazujemy na istniejącym dim_customer i symulujemy aktualizację

dim_customer_scd2 = dim_customer.copy()
dim_customer_scd2["valid_from"]  = pd.Timestamp("2010-01-01")
dim_customer_scd2["valid_to"]    = pd.Timestamp("9999-12-31")  # brak daty końcowej = aktualny
dim_customer_scd2["is_current"]  = True

# ── Symulacja zmiany kraju dla wybranego klienta ─────────────────────────────
# Przyjmijmy, że klient CustomerID=12346 zmienił kraj z 'United Kingdom' na 'Germany'
# w dniu 2011-06-01.

def apply_scd2_update(dim_df, customer_id, new_country, change_date):
    """Aktualizuje DimCustomer_SCD2 zgodnie z logiką SCD2."""
    mask = (dim_df["CustomerID"] == customer_id) & (dim_df["is_current"] == True)

    if mask.sum() == 0:
        print(f"Klient {customer_id} nie istnieje lub nie ma aktualnego rekordu.")
        return dim_df

    idx = dim_df[mask].index[0]
    old_country = dim_df.loc[idx, "Country"]

    if old_country == new_country:
        print(f"Brak zmiany – klient {customer_id} nadal w: {old_country}")
        return dim_df

    # Zamknij stary rekord
    dim_df.loc[idx, "valid_to"]   = pd.Timestamp(change_date) - pd.Timedelta(days=1)
    dim_df.loc[idx, "is_current"] = False

    # Dodaj nowy rekord
    new_key = dim_df["CustomerKey"].max() + 1
    new_row = {
        "CustomerKey": new_key,
        "CustomerID":  customer_id,
        "Country":     new_country,
        "valid_from":  pd.Timestamp(change_date),
        "valid_to":    pd.Timestamp("9999-12-31"),
        "is_current":  True
    }
    dim_df = pd.concat([dim_df, pd.DataFrame([new_row])], ignore_index=True)
    print(f"Klient {customer_id}: '{old_country}' → '{new_country}' od {change_date}")
    return dim_df


# Sprawdź czy CustomerID=12346 istnieje w danych
sample_customers = dim_customer_scd2["CustomerID"].head(3).tolist()
test_customer = sample_customers[0]  # użyj pierwszego dostępnego klienta
old_ctry = dim_customer_scd2.loc[dim_customer_scd2["CustomerID"] == test_customer, "Country"].values[0]

# Symuluj zmianę kraju
new_ctry = "Germany" if old_ctry != "Germany" else "France"
dim_customer_scd2 = apply_scd2_update(
    dim_customer_scd2,
    customer_id  = test_customer,
    new_country  = new_ctry,
    change_date  = "2011-06-01"
)

# Pokaż oba rekordy tego klienta
print("\nOba rekordy zmienionego klienta (historia):")
print(dim_customer_scd2[dim_customer_scd2["CustomerID"] == test_customer].to_string(index=False))

Klient 17850: 'United Kingdom' → 'Germany' od 2011-06-01

Oba rekordy zmienionego klienta (historia):
 CustomerKey  CustomerID        Country valid_from            valid_to  is_current
           1       17850 United Kingdom 2010-01-01 2011-05-31 00:00:00       False
        4347       17850        Germany 2011-06-01 9999-12-31 00:00:00        True


In [15]:
# Podsumowanie wymiaru SCD2
print(f"DimCustomer_SCD2 – łączna liczba rekordów: {len(dim_customer_scd2)}")
print(f"Rekordy aktualne (is_current=True):  {dim_customer_scd2['is_current'].sum()}")
print(f"Rekordy historyczne (is_current=False): {(~dim_customer_scd2['is_current']).sum()}")
print("\nPodgląd wymiaru SCD2:")
dim_customer_scd2.head()

DimCustomer_SCD2 – łączna liczba rekordów: 4347
Rekordy aktualne (is_current=True):  4346
Rekordy historyczne (is_current=False): 1

Podgląd wymiaru SCD2:


,CustomerKey,CustomerID,Country,valid_from,valid_to,is_current
0,1,17850,United Kingdom,2010-01-01,2011-05-31 00:00:00,False
1,2,13047,United Kingdom,2010-01-01,9999-12-31 00:00:00,True
2,3,12583,France,2010-01-01,9999-12-31 00:00:00,True
3,4,13748,United Kingdom,2010-01-01,9999-12-31 00:00:00,True
4,5,15100,United Kingdom,2010-01-01,9999-12-31 00:00:00,True


## 8. Weryfikacja schematu gwiazdy

Poniżej krótka weryfikacja poprawności relacji: sprawdzamy, czy wszystkie klucze w tabeli faktów mają odpowiedniki w wymiarach.

In [16]:
# Sprawdzenie integralności referencyjnej
missing_customer = fact_sales["CustomerKey"].isna().sum()
missing_product  = fact_sales["ProductKey"].isna().sum()
missing_date     = (~fact_sales["DateKey"].isin(dim_date["DateKey"])).sum()

print("=== Weryfikacja integralności referencyjnej ===")
print(f"Brakujące CustomerKey w FactSales: {missing_customer}")
print(f"Brakujące ProductKey  w FactSales: {missing_product}")
print(f"DateKey bez odpowiednika w DimDate: {missing_date}")

print("\n=== Podsumowanie tabel ===")
print(f"DimCustomer:    {len(dim_customer):>6} wierszy")
print(f"DimProduct:     {len(dim_product):>6} wierszy")
print(f"DimDate:        {len(dim_date):>6} wierszy")
print(f"FactSales:      {len(fact_sales):>6} wierszy")
print(f"\nŁączny przychód: {fact_sales['Revenue'].sum():>12,.2f}")
print(f"Średni przychód na pozycję: {fact_sales['Revenue'].mean():>8,.2f}")

=== Weryfikacja integralności referencyjnej ===
Brakujące CustomerKey w FactSales: 0
Brakujące ProductKey  w FactSales: 0
DateKey bez odpowiednika w DimDate: 0

=== Podsumowanie tabel ===
DimCustomer:      4346 wierszy
DimProduct:       3665 wierszy
DimDate:           305 wierszy
FactSales:      392692 wierszy

Łączny przychód: 8,887,208.89
Średni przychód na pozycję:    22.63


## 9. Wnioski

**Podjęte decyzje projektowe:**
- Ziarno na poziomie pozycji faktury (InvoiceNo + StockCode) zapewnia maksymalną elastyczność analityczną.
- Każdy wymiar otrzymał sztuczny klucz (surrogate key), uniezależniając hurtownię od systemów źródłowych.
- DimDate zbudowano z atrybutami (rok, miesiąc, kwartał, dzień tygodnia), co przyspiesza typowe zapytania czasowe bez konieczności każdorazowego parsowania daty.
- SCD Typ 2 dla DimCustomer pozwala śledzić zmiany kraju klienta w czasie, zachowując pełną historię transakcji.

**Czego się nauczyłem:**
- Czyszczenie danych jest kluczowym etapem ETL – każdy błędny rekord wpływa na jakość analiz.
- Wybór ziarna to jedna z najważniejszych decyzji projektowych: determinuje, jakie pytania hurtownia może odpowiedzieć.
- SCD2 jest potężniejszy od SCD1, ale generuje więcej rekordów i wymaga obsługi flagi `is_current` w zapytaniach.

**Trudności:**
- Jeden produkt (StockCode) może mieć różne opisy w danych – zastosowano `drop_duplicates(subset=["StockCode"])` zachowując pierwsze wystąpienie.
- Konwersja dat wymagała ustawienia odpowiedniego kodowania pliku CSV (ISO-8859-1).